# 05 Streaming algorithms against exact answers

`python -m etl.streaming_algos`. Stream: enwiki page-change events in publish order, 2026-10-19 23:00 to 2026-11-13 23:59 UTC. HyperLogLog and Flajolet-Martin (PCSA) for distinct pages and editors (E4), a Bloom filter on meta_id over the raw Bronze lines, DGIM for revert events in the last hour, reservoir sampling and the AMS second moment. Memory of the exact answer is an estimate (8 bytes per distinct hash).

In [1]:
import json, os, sys
from pathlib import Path
REPO = Path.cwd().resolve().parents[1] if Path.cwd().name == 'final' else Path.cwd()
sys.path.insert(0, str(REPO))
os.environ.setdefault('TRUTHLAG_HW2_DIR', str(REPO / 'ec-hw2'))
import pandas as pd
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 20)
from deltalake import DeltaTable
LAKE = REPO / 'data' / 'lake'
RUNS = LAKE / '_runs'
def run(name): return json.loads((RUNS / name).read_text())
def table(name, version=None, columns=None):
    return DeltaTable(str(LAKE / name), version=version).to_pandas(columns=columns)
def history(name, n=10):
    h = DeltaTable(str(LAKE / name)).history(n)
    return pd.DataFrame([{k: e.get(k) for k in ('version', 'operation', 'step', 'as_of', 'snapshot', 'rows')} for e in h])
d = run('streaming_algos.json')
d['stream'], d['exact_distinct']

({'events': 4617893,
  'first': '2026-10-19 23:00:00.504000+00:00',
  'last': '2026-11-13 23:59:59.844000+00:00',
  'revert_events': 170315},
 {'pages': 2153428, 'editors': 229828})

## Distinct counts: error and memory (5 hash salts per sketch)

In [2]:
pd.DataFrame(d['distinct'])

,item,sketch,estimate_salt0,exact,rel_error_salt0_pct,mean_rel_error_5_salts_pct,sd_rel_error_5_salts_pct,max_abs_error_5_salts_pct,theory_se_pct,sketch_bytes,exact_bytes_est
0,pages,HLL p=10,2163285,2153428,0.458,1.601,2.944,5.622,3.25,768,17227424
1,pages,HLL p=12,2169372,2153428,0.740,0.597,1.203,1.524,1.62,3072,17227424
2,pages,HLL p=14,2148428,2153428,-0.232,-0.296,1.184,2.357,0.81,12288,17227424
3,pages,FM-PCSA m=64,2304677,2153428,7.024,2.047,10.211,15.666,9.75,256,17227424
4,pages,FM-PCSA m=256,2224966,2153428,3.322,5.346,5.167,10.259,4.88,1024,17227424
5,pages,FM-PCSA m=1024,2156754,2153428,0.154,2.365,2.879,7.169,2.44,4096,17227424
6,editors,HLL p=10,219584,229828,-4.457,0.112,2.596,4.457,3.25,768,1838624
7,editors,HLL p=12,225641,229828,-1.822,-0.555,1.604,2.606,1.62,3072,1838624
8,editors,HLL p=14,225220,229828,-2.005,-0.737,1.161,2.005,0.81,12288,1838624
9,editors,FM-PCSA m=64,264175,229828,14.945,3.297,12.255,14.945,9.75,256,1838624


In [3]:
pd.DataFrame(d['spark_approx_count_distinct'])

,rsd,pages,editors,seconds
0,0.05,2198477,220940,4.6
1,0.01,2153746,229227,1.9


In [4]:
d['editors_merged_daily_hll12']

{'estimate': 225641,
 'exact': 229828,
 'rel_error_pct': -1.822,
 'sum_of_daily_exact': 466959}

## Bloom filter on meta_id

In [5]:
pd.DataFrame(d['bloom_meta_id'])

,target_fp_rate,bits,hashes,bloom_bytes,events,exact_repeats,flagged,true_repeats_found,false_positives,false_negatives,observed_fp_rate,exact_set_bytes_est,seconds
0,0.010,44262813,7,5532852,4617897,4,7566,4,7562,0,0.001638,36943144,8.8
1,0.001,66394219,10,8299278,4617897,4,591,4,587,0,0.000127,36943144,11.4


## DGIM: revert events in the last hour

In [6]:
pd.DataFrame(d['dgim_reverts_last_hour'])

,scope,r,eval_points,mean_rel_error_pct,p95_rel_error_pct,max_rel_error_pct,bound_pct,max_buckets,dgim_bytes_est,exact_window_bytes_est,share_exact_pct
0,wiki-wide,2,3599,11.25,24.10,40.13,50.0,17,136.0,7024.0,NaN
1,wiki-wide,4,3599,5.24,10.84,16.32,25.0,31,248.0,7024.0,NaN
2,per page (200 pages with most reverts),2,34634,41.18,50.00,50.00,50.0,9,NaN,NaN,5.04


## Reservoir sampling and AMS

In [7]:
d['reservoir_edits'], d['ams_f2_edits_per_page']

({'k': 10000,
  'stream_edits': 4209083,
  'exact_revert_event_share_pct': 4.046,
  'sample_share_pct': 3.76,
  'sample_95ci_pct': [3.387, 4.133]},
 [{'variables': 100,
   'estimate': 50088088,
   'exact_f2': 249630183,
   'rel_error_pct': -79.94},
  {'variables': 1000,
   'estimate': 215925958,
   'exact_f2': 249630183,
   'rel_error_pct': -13.5}])